# Backup and verified recovery

Back up a sample configuration, restore it, and test how the script handles a damaged archive.

Use **Runtime → Run all** in Google Colab. Each run uses a fresh temporary lab directory. The output below was captured on Linux.

[View the project on GitHub](https://github.com/Sehaj64/Portfolio/tree/main/projects/linux-operations/backup)

## Prepare the lab

In [1]:
%%bash
set -eu
lab=$(mktemp -d)
printf '%s' "$lab" > backup.lab
cd "$lab"
cat > backup.sh <<'BASH_FILE'
#!/usr/bin/env bash
# Create and verify a compressed full backup; retain only this tool's old archives.
set -euo pipefail
umask 077
[[ $# == 2 ]] || { echo "Usage: $0 SOURCE_DIRECTORY BACKUP_DIRECTORY" >&2; exit 1; }
source_dir=$(realpath -e -- "$1")
[[ -d $source_dir && $source_dir != / ]] || { echo 'Source must be a non-root directory' >&2; exit 1; }
mkdir -p -- "$2"
backup_dir=$(realpath -e -- "$2")
[[ $backup_dir != "$source_dir" && $backup_dir != "$source_dir/"* ]] || { echo 'Backups must be outside the source directory' >&2; exit 1; }
KEEP=${KEEP:-7}
[[ $KEEP =~ ^[1-9][0-9]?$ ]] || { echo 'KEEP must be 1..99' >&2; exit 1; }
exec 9>"$backup_dir/.backup.lock"
flock -n 9 || { echo 'Backup already running' >&2; exit 1; }
tmp=$(mktemp "$backup_dir/.pending.XXXXXX")
trap 'rm -f -- "$tmp"' EXIT
name="ops-backup-$(date -u +%Y%m%dT%H%M%S)-${tmp##*.}.tar.gz"
tar -czf "$tmp" -C "$source_dir" .
tar -tzf "$tmp" > /dev/null
mv -- "$tmp" "$backup_dir/$name"
(cd "$backup_dir" && sha256sum -- "$name" > "$name.sha256")
# Sort controlled names only; never remove unrelated archives or arbitrary user files.
mapfile -t archives < <(find "$backup_dir" -maxdepth 1 -type f -name 'ops-backup-*.tar.gz' -printf '%f\n' | LC_ALL=C sort -r)
for ((i=KEEP;i<${#archives[@]};i++)); do
  rm -- "$backup_dir/${archives[i]}"
  rm -f -- "$backup_dir/${archives[i]}.sha256"
done
printf 'BACKUP_OK archive=%s retained=%s\n' "$backup_dir/$name" "$KEEP"
BASH_FILE
cat > restore.sh <<'BASH_FILE'
#!/usr/bin/env bash
# GNU tar restore for trusted lab backups: fail closed on unsafe archive members.
set -euo pipefail
umask 077
export LC_ALL=C
[[ $# == 2 ]] || { echo "Usage: $0 ARCHIVE DESTINATION" >&2; exit 1; }
archive=$(realpath -e -- "$1")
destination=$(realpath -m -- "$2")
[[ ! -e $destination && ! -L $destination ]] || { echo 'Destination must not exist' >&2; exit 1; }
read -r expected _ < "$archive.sha256"
[[ $expected =~ ^[[:xdigit:]]{64}$ ]] || { echo 'Invalid checksum sidecar' >&2; exit 1; }
actual=$(sha256sum -- "$archive"); actual=${actual%% *}
[[ ${expected,,} == "$actual" ]] || { echo 'Checksum mismatch' >&2; exit 1; }
parent=$(dirname -- "$destination")
mkdir -p -- "$parent"
lock_id=$(printf '%s' "$destination" | sha256sum); lock_id=${lock_id%% *}
exec 9>"$parent/.restore-$lock_id.lock"
flock -n 9 || { echo 'Restore already running' >&2; exit 1; }
stage=$(mktemp -d "$parent/.restore-stage.XXXXXX")
trap 'rm -rf -- "$stage"' EXIT
# Preserve absolute names during listing so they can be rejected explicitly.
# Escaped/newline/non-ASCII names are unsupported by this deliberately narrow lab format.
tar --absolute-names --quoting-style=escape -tzf "$archive" > "$stage/names"
tar --absolute-names --quoting-style=escape -tvzf "$archive" > "$stage/types"
while IFS= read -r name; do
  case "$name" in
    /*|*\\*|..|../*|*/../*|*/..) echo 'Unsafe or unsupported archive path' >&2; exit 1;;
  esac
done < "$stage/names"
awk 'substr($0,1,1)!="-" && substr($0,1,1)!="d" {exit 1}' "$stage/types" || { echo 'Links and special files rejected' >&2; exit 1; }
mkdir "$stage/data"
tar -xzf "$archive" -C "$stage/data" --keep-old-files --no-same-owner --no-same-permissions
# Do not carry privileged mode bits into recovered files.
find "$stage/data" -type f -exec chmod u-s,g-s {} +
[[ ! -e $destination && ! -L $destination ]] || { echo 'Destination appeared during restore' >&2; exit 1; }
mv -T -n -- "$stage/data" "$destination"
[[ ! -e $stage/data ]] || { echo 'Destination already exists' >&2; exit 1; }
printf 'RESTORE_OK destination=%s\n' "$destination"
BASH_FILE
echo 'Lab scripts ready.'

Lab scripts ready.


## Create a backup

In [2]:
%%bash
set -eu
cd "$(cat backup.lab)"
mkdir -p source
printf 'service=reports\nport=8080\n' > source/settings.conf
bash backup.sh source archives

BACKUP_OK archive=/tmp/tmp.tRRoTl6fcw/archives/ops-backup-20261127T075629-bnJbTo.tar.gz retained=7


## Restore and compare the files

In [3]:
%%bash
set -eu
cd "$(cat backup.lab)"
archive=$(find archives -name '*.tar.gz' -print -quit)
bash restore.sh "$archive" restored
cmp source/settings.conf restored/settings.conf
cat restored/settings.conf
echo 'Restored file matches the original.'

RESTORE_OK destination=/tmp/tmp.tRRoTl6fcw/restored
service=reports
port=8080
Restored file matches the original.


## Try restoring a damaged copy

In [4]:
%%bash
set -eu
cd "$(cat backup.lab)"
archive=$(find archives -name '*.tar.gz' -print -quit)
cp "$archive" damaged.tar.gz
cp "$archive.sha256" damaged.tar.gz.sha256
printf x >> damaged.tar.gz
if bash restore.sh damaged.tar.gz rejected; then exit 1; fi
test ! -e rejected
echo 'Damaged copy rejected. No restore directory was created.'

Damaged copy rejected. No restore directory was created.
Checksum mismatch
